# IE500618 – Assignment 2: Diabetes Classification

**Group members:** Fillip Grüner, Kacper Fendrykowski and Julianne Mandal

The goal of this notebook is to classify survey respondents as having diabetes or not, using the supplied dataset (`diabetes_binary_classification_data.csv`). The goal is not to get the highest possible accuracy. We focus on understanding the data, keeping the test set protected, fitting a Logistic Regression correctly and interpreting the predictions and errors. We follow the workflow from the assignment:

**Understand → EDA → Split → Prepare → Fit → Predict → Evaluate → Interpret**

Each step contains the code and outputs, together with short explanations of what we found and why we made our decisions. When the meaning of a variable was unclear, we used the *Diabetes Classification – Understanding the Dataset* guide.

In [ ]:
# importing necessary libs
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score


sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)


In [ ]:
# reading csv file, saving in variable 'df'
df = pd.read_csv('../data/diabetes_binary_classification_data.csv')

df.head()

# Sandbox for testing stuff **NOT PART OF FINAL SUBMISSION**

In [ ]:
df.info()

In [ ]:
# heatmap for correlation between all categories
plt.figure(figsize=(40, 24))
sns.heatmap(data=df.corr(numeric_only=True), cmap="coolwarm", annot=True, cbar=True)

In [ ]:
df['Veggies'].describe()

In [ ]:
df['Age'].value_counts()

# Step 1 - Understand the data


- *Identify the input features X and the target y.*
- *Explain what class 0 and class 1 mean.*
- *Check the number and proportion of observations in each class.*
- *Choose three features of different types and briefly explain what their values mean. Use the Understanding the Dataset guide.*

### Input features X and target y

In [ ]:
df.info()

**Findings:** The dataset contains **253,680 survey respondents (rows)** and 22 columns: the target and 21 other columns. There are no missing values. All columns are stored as numbers, but they do not all represent numerical measurements, so we use the guide to see what each column means.

The target `y` is `Diabetes_binary`, and the input features `X` are the other 21 columns.The target y is Diabetes_binary, and the input features X is the other 21 columns.

### Meaning of class 0 and class 1

**Class 0** means no diabetes, and **class 1** means prediabetes or diabetes.

Because class 1 combines two conditions, the model predicts the supplied binary outcome and cannot tell prediabetes and diabetes apart.

### Number and proportion of observations

In [ ]:
df['Diabetes_binary'].value_counts()

In [ ]:
df['Diabetes_binary'].value_counts(normalize=True)

**Findings:**

- 218,334 respondents (86.1%) are in class 0, and 35,346 (13.9%) are in class 1. The target is therefore **imbalanced**.
- This matters for the evaluation. A model that answers "no diabetes" for every single respondent would still score about 86% accuracy without finding a single person with diabetes. Accuracy alone can therefore hide poor detection of class 1, so we also look at precision and recall in Step 6.

### Three features of different types and brief explanation of value meanings

| Feature | Type | What the values mean |
|---|---|---|
| `BMI` | Numerical measurement | A person's measured Body Mass Index. The values in the CSV range from 12 to 98. |
| `NoDocbcCost` | Binary categorical | Whether the respondent needed to see a doctor in the past 12 months but could not because of cost. 0 = no, 1 = yes. |
| `Age` | Ordinal categorical | An age group from 1 to 13, where 1 = 18–24, 2 = 25–29, … , 9 = 60–64, … , 13 = 80+ years. |

The numbers stored in the CSV file are not enough to understand a feature:

- `BMI = 30` is a real measured value, but `Age = 9` is a group code. It means the age group 60–64, not an age of 9 years.
- The groups in `Age` have an order, but the codes are not measured distances.
- `NoDocbcCost = 1` only makes sense when we know the question behind it. It does not mean "more" of something, it means "yes".
- `Sex` is also binary, but it is coded 0 = female and 1 = male, so it is **not** a yes/no question like the other binary features.

We therefore used the guide to decide how each feature should be explored and prepared.

# Step 2 - Explore the data


*Use EDA to investigate all input features and their relationships with Diabetes_binary. Choose plots or tables that are appropriate for each type of feature:*

- *for numerical features, examine their distributions and compare them between the two target classes;*  
- *for binary categorical features, examine counts/proportions and compare the proportion of Diabetes_binary across the two categories;* 
- *for ordered categorical features, examine how the distribution or proportion of Diabetes_binary changes across the ordered categories.*

*Explore all input features, but focus your detailed interpretation on the most informative or interesting results. You may use loops or functions where appropriate rather than repeating essentially identical code. For the results you discuss in detail, state the question, show appropriate evidence, explain what you observe, and explain what the evidence allows you to conclude and what it does not. Look for patterns such as differences between the two target classes, changes in the proportion of class 1 across categories, unusual distributions, and features that appear to have little relationship with the target. Remember: EDA describes associations and patterns in the observed data; an association does not show that one variable causes diabetes.* 

The guide separates the input features into three types, so we use a different method for each type:

| Type | Features | How we explore them |
|---|---|---|
| **Numerical** | `BMI`, `MentHlth`, `PhysHlth` | distributions compared between the two classes |
| **Binary categorical** | 14 features (13 yes/no and `Sex`) | share of class 1 in each of the two categories |
| **Ordinal categorical** | `GenHlth`, `Age`, `Education`, `Income` | share of class 1 across the ordered categories |

We do not calculate ordinary correlations for the category codes, since the codes are not measured amounts. EDA only describes **associations** in the observed data. It does not show that one feature causes diabetes.

In [ ]:
binary = ["HighBP", "HighChol", "CholCheck", "Smoker", "Stroke", "HeartDiseaseorAttack",
          "PhysActivity", "Fruits", "Veggies", "HvyAlcoholConsump", "AnyHealthcare",
          "NoDocbcCost", "DiffWalk", "Sex"]
numerical = ["BMI", "MentHlth", "PhysHlth"]
ordinal = ["GenHlth", "Age", "Education", "Income"]

### 2a) Numerical features

`BMI` is a measurement, while `MentHlth` and `PhysHlth` count the number of days in the past 30 days with poor mental or physical health. We compare them between the two classes with box plots and look at the group means.

In [ ]:
for col in numerical:
    sns.boxplot(x="Diabetes_binary", y=col, data=df)
    plt.title(f"{col} by Diabetes_binary")
    plt.show()

df.groupby("Diabetes_binary")[numerical].mean().round(1)

**Findings:**

- **`BMI`** is higher for respondents with diabetes: the mean is 31.9 for class 1 compared with 27.8 for class 0.
- **`PhysHlth`** also differs clearly: respondents with diabetes report 8.0 days of poor physical health on average, compared with 3.6 days for class 0.
- **`MentHlth`** shows a smaller difference (4.5 vs 3.0 days).
- For `MentHlth` and `PhysHlth` the median is 0 (or 1) in both classes, because most respondents report no bad days. The boxes are therefore squeezed together, and the means are more informative than the boxes here.

This shows that the two groups differ in these variables, but not that a high BMI causes diabetes.

### 2b) Binary categorical features

There are 14 binary features: 13 yes/no questions and `Sex` (0 = female, 1 = male). For each feature we calculate the share of respondents with class 1 in category 0 and in category 1, and sort the features by the share in category 1.

In [ ]:
rates = pd.DataFrame({c: df.groupby(c)["Diabetes_binary"].mean() * 100 for c in binary}).T
rates.columns = ["Value 0", "Value 1"]   # 0 = no (Sex: female), 1 = yes (Sex: male)
rates.sort_values("Value 1").plot(kind="barh", figsize=(8, 7))
plt.xlabel("Diabetes rate (%)")
plt.title("Diabetes rate by binary feature")
plt.show()

**Findings:**

- The largest differences are found for `HeartDiseaseorAttack` (12.0% vs 33.0%), `Stroke` (13.2% vs 31.8%), `DiffWalk` (10.5% vs 30.7%), `HighBP` (6.0% vs 24.4%) and `HighChol` (8.0% vs 22.0%). Respondents with these conditions are much more likely to be in class 1.
- `PhysActivity` goes in the other direction: the share is lower for respondents who are physically active (11.6% vs 21.1%).
- Some features show little relationship with the target, for example `Sex` (13.0% for females vs 15.2% for males), `Fruits` (15.8% vs 12.9%) and `AnyHealthcare` (11.5% vs 14.1%).
- Two results are **surprising**. `HvyAlcoholConsump` has a *lower* share for heavy drinkers (5.8% vs 14.4%), and `CholCheck` has a *higher* share for respondents who have had their cholesterol checked (14.4% vs 2.5%).

These are associations only. The alcohol result does not mean that drinking protects against diabetes. People who already have diabetes may drink less, or other factors such as age may play a role. In the same way, people with diabetes are probably more likely to have had a cholesterol check.

### 2c) Ordered categorical features

`GenHlth`, `Age`, `Education` and `Income` are ordered categories. We therefore look at how the share of class 1 changes across the ordered categories, using bar plots.

In [ ]:
for col in ordinal:
    (df.groupby(col)["Diabetes_binary"].mean() * 100).plot(kind="bar", figsize=(8, 4))
    plt.title(f"Diabetes rate (%) by {col}")
    plt.ylabel("Diabetes rate (%)")
    plt.show()

**Findings:**

- **`GenHlth`** shows the clearest pattern. The share of class 1 rises steadily from 2.5% (excellent) to 37.9% (poor).
- **`Age`** rises from 1.4% (18–24) to about 21–22% for the groups 70–74 and 75–79, and then drops slightly to 18.5% for 80+.
- **`Income`** and **`Education`** go in the opposite direction: the share falls from 24.3% in the lowest income group to 8.0% in the highest, and from about 27–29% for the lowest education groups to 9.7% for college graduates.

Poorer self-rated health, higher age and lower income or education are associated with diabetes in this dataset. We cannot separate the effects of the features from the plots alone, because they are related to each other (for example age and health). `GenHlth` is also self-rated, so respondents who already have diabetes may rate their health lower.

### Summary of Step 2

- The most informative features are `GenHlth`, `Age`, `HighBP` and `BMI`, which all show a clear difference between the classes or a clear trend across the categories.
- The heart disease, stroke and difficulty-walking features also show large differences.
- `HvyAlcoholConsump` is an interesting result, because the direction is the opposite of what we might expect. It shows why associations must be interpreted with care.
- Some features, such as `Sex`, `Fruits` and `AnyHealthcare`, have little relationship with the target.
- All results describe associations in the observed data. None of them show that a feature causes diabetes.

# Step 3 - Create TRAIN and TEST


- *Split X and y into TRAIN and TEST.* 
- *Use stratify=y so that TRAIN and TEST have approximately the same proportion of class 0 and class 1.*  
- *Keep TEST separate until evaluation. Do not balance or resample TEST.*

Before we prepare any features or fit any model, we split the data into a training set (TRAIN) and a test set (TEST). We use 80% for training and 20% for testing. The most important rule in the rest of the notebook is:

> **TEST is kept separate until the final evaluation. Nothing is learned from it, and it is never balanced or resampled.**

We use `stratify=y` so that TRAIN and TEST have approximately the same share of class 0 and class 1. Without it, the small class 1 could be unevenly distributed between the two sets by chance.

In [ ]:
X = df.drop(columns="Diabetes_binary")
y = df["Diabetes_binary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=13)

print(X_train.shape, X_test.shape)
print("Class 1 share TRAIN:", y_train.mean().round(3), " TEST:", y_test.mean().round(3))

**Findings:** TRAIN contains 202,944 respondents and TEST contains 50,736. The share of class 1 is 13.9% in both sets, so the stratified split worked as intended.

**Why TEST must stay separate:** TEST is meant to act like new, unseen respondents. If TEST were used to fit the preprocessing or the model, information from the test respondents would leak into the model, and the test results would be too optimistic. We would then no longer have an honest estimate of how the model performs on new data.

# Step 4 - Prepare the features


*Prepare the input features using the information in the Understanding the Dataset guide and the preprocessing methods introduced in the course. Your preprocessing choice should reflect whether a feature is numerical, binary categorical, or ordered categorical. Briefly justify how you represent each type There are no missing values in this dataset, so missing - value imputation is not required.*

- *Use a Pipeline/ColumnTransformer where appropriate.* 
- *Anything learned from the data during preprocessing must be learned from TRAIN only and then applied to TEST.*

In this step we decide how each feature is represented before it goes into the Logistic Regression. The choice depends on the type of feature, based on the guide:

| Type | Features | Representation | Why |
|---|---|---|---|
| **Numerical** | `BMI`, `MentHlth`, `PhysHlth` | standardized (mean 0, standard deviation 1) | real measurements and counts on very different scales |
| **Ordinal** | `GenHlth`, `Age`, `Education`, `Income` | kept as ordered numbers, then standardized | the order carries meaning (a higher code is a higher group) |
| **Binary** | 13 yes/no features and `Sex` (0 = female, 1 = male) | no change (passthrough) | already coded 0/1, so no further encoding is needed |

The ordinal codes are treated as ordered categories, not as measured amounts. Keeping them as ordered numbers is a simple choice that keeps the order, and it uses only one column per feature.

There are no missing values in the dataset, so no imputation is needed. The preparation is built with a `ColumnTransformer`, which is placed inside a `Pipeline` in Step 5. The scaler therefore learns its mean and standard deviation from **TRAIN only**, and the same values are then applied to TEST.

In [ ]:
preprocess = ColumnTransformer([
    ("scale", StandardScaler(), numerical + ordinal),
    ("binary", "passthrough", binary),
])

# Step 5 - Train Logistic Regression


*Use Logistic Regression, as introduced in the classification lecture. Fit the complete Pipeline using TRAIN.*  

- *Use model.predict(X_test) to obtain predicted classes.* 
- *Use model.predict_proba(X_test)[:, 1] to obtain predicted probabilities for class 1.* 

We combine the preprocessing from Step 4 and a `LogisticRegression` in one `Pipeline` and fit the whole pipeline on TRAIN. This means that the scaler and the model both learn from TRAIN only. We then use the fitted model on TEST in two ways:

- `model.predict(X_test)` gives the **predicted class** (0 or 1) for each respondent.
- `model.predict_proba(X_test)[:, 1]` gives the **predicted probability of class 1**.

By default, `predict` turns a probability into class 1 when it is at least 0.50. In Step 7 we use the probabilities to try a different threshold.

In [ ]:
# chains the preprocessing and the model, so the scaler and the model are both fitted on TRAIN only
model = Pipeline([
    ("prep", preprocess),
    ("lr", LogisticRegression(max_iter=1000)),
])

# the only place anything is learned
model.fit(X_train, y_train)

y_pred = model.predict(X_test)              # holds the predicted classes (0 or 1) for each row in TEST
y_prob = model.predict_proba(X_test)[:, 1]  # holds the predicted probability of class 1 for each row in TEST

# Step 6 - Evaluate the model


*Evaluate the Logistic Regression model on TEST using:* 

- *confusion matrix;* 
- *accuracy;* 
- *precision;* 
- *recall;*
- *F1 score.* 

*Explain what a false positive and a false negative mean for this diabetes classification problem. Do not judge the model from accuracy alone.* 

Now we use TEST for the first time. We evaluate the model with the confusion matrix, accuracy, precision, recall and F1 score, and we do not judge it on accuracy alone.

For this problem:

- A **false positive (FP)** is a respondent *without* diabetes who is predicted as class 1.
- A **false negative (FN)** is a respondent *with* diabetes who is predicted as class 0.

The measures answer different questions:

- **Accuracy:** of all respondents, how many are classified correctly?
- **Precision:** of the respondents predicted as class 1, how many actually are class 1?
- **Recall:** of the respondents who actually are class 1, how many does the model find?
- **F1:** a single number that combines precision and recall.

In [ ]:
def evaluate(pred):
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")
    print("Accuracy :", round(accuracy_score(y_test, pred), 3))
    print("Precision:", round(precision_score(y_test, pred), 3))
    print("Recall   :", round(recall_score(y_test, pred), 3))
    print("F1       :", round(f1_score(y_test, pred), 3))

evaluate(y_pred)

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["No diabetes", "Prediabetes/diabetes"])
plt.title("Confusion matrix (threshold 0.50)")
plt.show()

**Findings (threshold 0.50):**

| | Predicted 0 | Predicted 1 |
|---|---|---|
| **Actual 0** | TN = 42,754 | FP = 913 |
| **Actual 1** | FN = 5,997 | TP = 1,072 |

Accuracy is 0.864, precision is 0.540, recall is 0.152 and F1 is 0.237.

- **Accuracy looks good, but it is misleading.** Most respondents are in class 0 (86%), and the model classifies 98% of them correctly. This gives a high accuracy even though class 1 is mostly missed.
- **Recall is low.** Of the 7,069 respondents with diabetes in TEST, the model finds only 1,072 and misses 5,997, which is about 85%.
- **Precision is moderate.** About 54% of the respondents that the model flags as class 1 actually have diabetes.
- For a screening problem, a **false negative** is a missed case, and the person may get advice or treatment later than they should. A **false positive** leads to unnecessary worry or testing. A false negative is usually considered the more serious error, so the low recall is the main weakness of this model.

### Summary of Step 6

- The model has an accuracy of 0.864, but this is mostly because of the large class 0.
- Recall (0.152) shows that the model finds only a small part of the people with diabetes.
- Accuracy alone is therefore not enough to judge the model, and precision and recall must be considered together.

# Step 7 - Probability and threshold


*Use the predicted probabilities from Logistic Regression to compare two decision thresholds:* 

- *Threshold 0.50* 
- *Threshold 0.40* 

*For each threshold, calculate the confusion matrix, precision and recall. Compare the results. Explain what happened to false positives, false negatives, precision and recall. Also explain what did NOT change when you changed only the threshold.*

A probability is not yet a class. The **threshold** decides when a predicted probability is turned into class 1. In Step 6 we used the default threshold of 0.50. Here we compare it with a lower threshold of 0.40, using the same predicted probabilities `y_prob` from the same fitted model. For each threshold we calculate the confusion matrix, accuracy, precision, recall and F1 score.

### Threshold 0.50

In [ ]:
print("--- Threshold 0.50 ---")
evaluate((y_prob >= 0.50).astype(int))

**Findings:** TN = 42,754, FP = 913, FN = 5,997 and TP = 1,072. These are exactly the same numbers as in Step 6, which confirms that `predict` uses a threshold of 0.50.

### Threshold 0.40

In [ ]:
print("--- Threshold 0.40 ---")
evaluate((y_prob >= 0.40).astype(int))

**Findings:** TN = 41,669, FP = 1,998, FN = 5,140 and TP = 1,929. More respondents are now flagged as class 1 than at threshold 0.50.

### Comparing the results

In [ ]:
results = []

for t in [0.50, 0.40]:
    pred = (y_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    results.append({
        "Threshold": t,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "False positives": fp,
        "False negatives": fn,
    })

threshold_table = pd.DataFrame(results).round(3)
threshold_table

**Findings:**

| Threshold | Accuracy | Precision | Recall | F1 | False positives | False negatives |
|---|---|---|---|---|---|---|
| 0.50 | 0.864 | 0.540 | 0.152 | 0.237 | 913 | 5,997 |
| 0.40 | 0.859 | 0.491 | 0.273 | 0.351 | 1,998 | 5,140 |

When the threshold is lowered from 0.50 to 0.40, respondents are flagged as class 1 at a lower probability, so **more respondents are flagged**:

- **False positives** increase (913 → 1,998), because more respondents without diabetes are flagged.
- **False negatives** decrease (5,997 → 5,140), because fewer respondents with diabetes are missed.
- **Recall** increases (0.152 → 0.273), because the model finds more of the real cases.
- **Precision** decreases (0.540 → 0.491), because a larger share of the flagged respondents do not have diabetes.
- Accuracy drops slightly (0.864 → 0.859), while F1 improves (0.237 → 0.351), because recall increases more than precision decreases.

**What did not change:** the model. We did not retrain anything, so the coefficients and the predicted probabilities are exactly the same. Only the cutoff for turning a probability into a class has changed. The actual classes in TEST (7,069 respondents with diabetes) are also the same.

### Summary of Step 7

- A lower threshold gives a trade-off: more cases are found, but more false alarms are created.
- False positives and false negatives move in opposite directions, so we cannot reduce both by changing only the threshold.
- Which threshold is better depends on how costly each type of error is. For screening, missing a case is often considered worse than an extra test, which would favour 0.40.
- Even at 0.40 the model still misses 5,140 of the 7,069 respondents with diabetes.

# Step 8 - Conclusion


*Summarize the most important things you learned from the EDA and the Logistic Regression results.* 
- *Give at least two limitations of this analysis.* 
- *Do not interpret an association as evidence that one feature causes diabetes.*

**What we learned from the EDA**

- Diabetes (class 1) is much more common among respondents with poorer self-rated health (2.5% for excellent vs 37.9% for poor), older age groups (1.4% for 18–24 vs about 21–22% for 70–79), high blood pressure (24.4% vs 6.0%) and a higher BMI (mean 31.9 vs 27.8).
- The share of class 1 is also higher for respondents with lower income and less education.
- Some features show little relationship with the target, for example `Sex` and `Fruits`.
- `HvyAlcoholConsump` was a surprising result (5.8% for heavy drinkers vs 14.4% for the others). It is unlikely to mean that alcohol protects against diabetes, and it shows why associations must be interpreted with care.

**What we learned from the Logistic Regression model**

- At threshold 0.50 the model reaches an accuracy of 0.864, but the recall is only 0.152. It finds about 15% of the respondents with diabetes and misses 5,997 of 7,069 in TEST.
- The accuracy is high mostly because 86% of the respondents are in class 0, so accuracy alone is not enough to judge the model.
- Lowering the threshold to 0.40 increases recall (0.152 → 0.273) and F1 (0.237 → 0.351), but also increases false positives (913 → 1,998) and lowers precision (0.540 → 0.491). The threshold changes the balance between the two types of errors, not the model itself.

**Limitations**

1. **Self-reported survey data:** The answers come from a telephone survey and may be inaccurate or biased, for example for self-rated health, alcohol consumption or physical activity.
2. **The target combines two conditions:** Class 1 contains both prediabetes and diabetes, so the model cannot tell them apart.
3. **Association, not causation:** Both the EDA and the model show patterns in the observed data. They do not show that any feature causes diabetes, and many features (age, income, health) are related to each other.
4. **Simple setup:** We used one model, one train/test split and the default handling of the class imbalance. The results are therefore not necessarily the best that the data can offer.